### Architecture 3 — Product & sales multi agentic bot

In [1]:
from typing import (
    TypedDict,
    Literal,
    Annotated,
)

import operator
import os
from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
)
from langchain_core.tools import tool
# from langchain.agents import create_agent

from langgraph.graph import (
    StateGraph,
    START,
    END,
    MessagesState,
)
from langgraph.graph.message import add_messages
from langgraph.types import (
    Command,
    Send,
)
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

True

In [3]:
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found - add it to your .env file.")
    
model = ChatOpenAI(
    model="nvidia/nemotron-3.5-lightning:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    max_retries=3,   # free models hit 429 rate limits often
    timeout=120,
)

In [ ]:
# step 1: one team state

class TeamState(TypedDict):
    task: str
    sales_output: str
    support_output: str
    message: Annotated[list, add_messages]

In [ ]:
# def make_agent(role, action, key):
#     def agent(state):
#         result = model.invoke(f"Your are a {role} specialist. Your task is to {action}. The current task is: {state['task']}. Please provide your output.")
#         return {key: result}
#     return agent

In [ ]:
# step 2: creating sales team
def sales_agent(state: TeamState):
    task = state["task"]

    product_agent = model.invoke(
        f"""You are the Product Agent. Answer product questions using only the
provided information. Never guess; say when details are unavailable.

Customer question: {task}"""
    )

    pricing_agent = model.invoke(
        f"""You are the Pricing Agent. Answer pricing questions using only the
provided information. Never guess; say when details are unavailable.

Customer question: {task}
Product information: {product_agent.content}"""
    )

    offer_agent = model.invoke(
        f"""You are the Offer Agent. Answer questions about offers using only the
provided information. Never guess; say when details are unavailable.

Customer question: {task}
Product information: {product_agent.content}
Pricing information: {pricing_agent.content}"""
    )

    return {
        "sales_output": (
            f"Product Agent: {product_agent.content}\n"
            f"Pricing Agent: {pricing_agent.content}\n"
            f"Offer Agent: {offer_agent.content}"
        )
    }

In [ ]:
def support_agent(state: TeamState):
    task = state["task"]

    delivery_agent = model.invoke(
        f"""You are the Delivery Agent. Answer delivery questions using only the
provided information. Never guess; say when details are unavailable.

Customer question: {task}"""
    )

    refund_agent = model.invoke(
        f"""You are the Refund Agent. Assess refund eligibility using the provided
refund policy and customer information. Never guess; say when details are unavailable.

Customer question: {task}"""
    )

    returns_agent = model.invoke(
        f"""You are the Returns Agent. Answer return questions using only the
provided information. Never guess; say when details are unavailable.

Customer question: {task}"""
    )

    return {
        "support_output": (
            f"Delivery Agent: {delivery_agent.content}\n"
            f"Refund Agent: {refund_agent.content}\n"
            f"Returns Agent: {returns_agent.content}"
        )
    }